# Testing-Sample Conflict and Pattern Analysis Notebook

This notebook is the **end-to-end analysis pipeline** for the testing-sample conflict study.

It is designed to run in **Google Colab** with this split source setup:

- the **results workbook** is read from the **uploaded Colab content** (`/content` in Colab; `/mnt/data` in this environment)
- the **Testing Sample assignment files** are read from Google Drive at:

  - `AI_RSRCH_FILES/Assignments Part/Assignments Set/Testing Sample`

The notebook does all of the following:

1. locates and validates the uploaded testing-sample workbook  
2. loads the full benchmark results  
3. rebuilds derived binary and soft metrics when needed  
4. maps each result row to its underlying text/code file  
5. computes conflict and non-conflict indicators  
6. quantifies conflict by language, label, and tool  
7. detects disagreement patterns, lone dissenters, and split signatures  
8. compares conflicting vs non-conflicting files on measurable features  
9. exports all tables and figures for the paper

In [ ]:
# If running in Colab, mount Google Drive for the assignment-set folder.
# The results workbook itself will be read from uploaded Colab content, not from Drive.

try:
    from google.colab import drive

    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Colab. Using local paths as configured below.")

In [ ]:
# Core imports and package installation.
# The installs are intentionally explicit so the notebook is reproducible in a fresh Colab runtime.

import sys
import subprocess
import pkgutil


def ensure_package(pkg_name, import_name=None):
    import_name = import_name or pkg_name
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])


for pkg, imp in [
    ("openpyxl", "openpyxl"),
    ("xlsxwriter", "xlsxwriter"),
    ("scipy", "scipy"),
    ("python-docx", "docx"),
    ("tqdm", "tqdm"),
    ("chardet", "chardet"),
    ("openai", "openai"),
]:
    ensure_package(pkg, imp)

import os
import re
import math
import zipfile
import shutil
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 1. Configuration

The next cell uses:

- **Google Drive** only for the `Testing Sample` assignment-set folder
- **uploaded Colab content** for the benchmark workbook

So you should upload the results workbook into the Colab session, while keeping the assignment files in Drive.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("CONFLICT_ANALYSIS_DATA_ROOT", "/mnt/data"))
PROJECT_ROOT = Path(
    os.environ.get("CONFLICT_ANALYSIS_PROJECT_ROOT", DATA_ROOT / "conflict_analysis")
)
TESTING_SAMPLE_DIR = Path(
    os.environ.get(
        "CONFLICT_ANALYSIS_TESTING_SAMPLE_DIR", PROJECT_ROOT / "testing_sample"
    )
)
OUTPUT_BASE_DIR = Path(
    os.environ.get("CONFLICT_ANALYSIS_OUTPUT_DIR", PROJECT_ROOT / "results")
)

UPLOAD_SEARCH_DIRS = [
    Path(path)
    for path in os.environ.get("CONFLICT_ANALYSIS_UPLOAD_DIRS", "/content").split(
        os.pathsep
    )
    if path
]
UPLOADED_RESULTS_FILENAME_HINTS = ["Testing_Sample_Detection_Results.xlsx"]

USE_OPENAI_QUAL_REVIEW = os.environ.get(
    "CONFLICT_ANALYSIS_USE_OPENAI_QUAL_REVIEW", "true"
).lower() in {"1", "true", "yes"}
OPENAI_API_KEY_ENV = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.environ.get("CONFLICT_ANALYSIS_OPENAI_MODEL", "gpt-4.1-mini")
OPENAI_MAX_FILES_PER_SUBSET = int(
    os.environ.get("CONFLICT_ANALYSIS_OPENAI_MAX_FILES_PER_SUBSET", "5")
)

RANDOM_SEED = int(os.environ.get("CONFLICT_ANALYSIS_RANDOM_SEED", "42"))

## 2. Helper functions

These helpers make the notebook robust to small changes in file names, column names, and workbook structure.

In [ ]:
# =========================
# HELPERS
# =========================


def norm_text(x):
    return "" if x is None else str(x).strip()


def norm_header(x):
    s = norm_text(x).lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")


def find_existing_project_root():
    if PROJECT_ROOT.exists():
        return PROJECT_ROOT
    for candidate in ALT_PROJECT_ROOTS:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find AI_RSRCH_FILES project root. "
        "Please verify the path in the configuration cell."
    )


def recursive_find_files(root, suffixes=None, name_contains=None):
    root = Path(root)
    if not root.exists():
        return []
    suffixes = None if suffixes is None else {s.lower() for s in suffixes}
    out = []
    for p in root.rglob("*"):
        if p.is_file():
            if suffixes is not None and p.suffix.lower() not in suffixes:
                continue
            if (
                name_contains is not None
                and name_contains.lower() not in p.name.lower()
            ):
                continue
            out.append(p)
    return sorted(out)


def pick_best_workbook(candidates):
    if not candidates:
        return None

    def score(p):
        name = p.name.lower()
        s = 0
        if "testing_sample_detection_results" in name:
            s += 50
        if "detection_results" in name:
            s += 20
        if "xlsx" in name:
            s += 10
        try:
            s += int(p.stat().st_mtime / 100000)
        except Exception:
            pass
        return s

    return sorted(candidates, key=score, reverse=True)[0]


def load_excel_sheets(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        try:
            df = pd.read_csv(path, dtype=object)
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="utf-8-sig", dtype=object)
        except Exception:
            df = pd.read_csv(path, encoding="latin1", dtype=object)
        return {path.stem: df}

    xls = pd.ExcelFile(path)
    sheets = {}
    for s in xls.sheet_names:
        sheets[s] = pd.read_excel(path, sheet_name=s, dtype=object)
    return sheets


def first_matching_col(df, aliases, required=False):
    colmap = {norm_header(c): c for c in df.columns}
    for a in aliases:
        na = norm_header(a)
        if na in colmap:
            return colmap[na]
    if required:
        raise KeyError(f"Could not find any of these columns: {aliases}")
    return None


def safe_num(x):
    return pd.to_numeric(x, errors="coerce")


def truth_class_from_label(label):
    s = norm_text(label)
    if s == "AI-Free":
        return "Human"
    if s in {"AI-Generated", "Humanized AI"}:
        return "AI"
    return None


def language_family(x):
    s = norm_text(x)
    if s in {"Arabic", "English", "Coding"}:
        return s
    return s


def detect_score_scale(series):
    vals = [v for v in pd.to_numeric(series, errors="coerce").dropna().tolist()]
    if not vals:
        return "percent_0_100"
    return "fraction_0_1" if max(vals) <= 1.0 else "percent_0_100"


def to_score_percent(raw_value, scale_mode):
    x = pd.to_numeric(raw_value, errors="coerce")
    if pd.isna(x):
        return np.nan
    if scale_mode == "fraction_0_1":
        x = x * 100.0
    x = float(max(0.0, min(100.0, x)))
    return x


def binary_prediction_from_score_or_text(score_percent, result_text):
    if pd.notna(score_percent):
        return "AI" if float(score_percent) >= BINARY_THRESHOLD else "Human"

    s = norm_text(result_text).lower()
    if s == "":
        return np.nan

    human_phrases = [
        "human",
        "likely human",
        "fully human",
        "human written",
        "human-written",
        "predominantly human",
    ]
    ai_phrases = [
        "ai",
        "likely ai",
        "ai generated",
        "generated by ai",
        "ai written",
        "ai-written",
        "machine generated",
        "fully ai",
    ]

    if any(p in s for p in human_phrases) and not any(
        p in s for p in ["likely ai", "ai generated", "generated by ai"]
    ):
        return "Human"
    if any(p in s for p in ai_phrases) and "human" not in s:
        return "AI"

    return np.nan


def soft_correctness_percent(score_percent, truth_class):
    if pd.isna(score_percent) or pd.isna(truth_class) or truth_class is None:
        return np.nan
    return float(score_percent) if truth_class == "AI" else float(100.0 - score_percent)


def safe_div(a, b):
    return np.nan if (b is None or b == 0 or pd.isna(b)) else a / b


def cliffs_delta(x, y):
    x = [float(v) for v in x if pd.notna(v)]
    y = [float(v) for v in y if pd.notna(v)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = lt = 0
    for xi in x:
        for yj in y:
            if xi > yj:
                gt += 1
            elif xi < yj:
                lt += 1
    return (gt - lt) / (len(x) * len(y))


def benjamini_hochberg(pvals):
    p = np.array([np.nan if pd.isna(v) else float(v) for v in pvals], dtype=float)
    out = np.full_like(p, np.nan)
    mask = ~np.isnan(p)
    if mask.sum() == 0:
        return out
    vals = p[mask]
    order = np.argsort(vals)
    ranked = vals[order]
    m = len(ranked)
    adj = np.empty(m, dtype=float)
    prev = 1.0
    for i in range(m - 1, -1, -1):
        rank = i + 1
        candidate = ranked[i] * m / rank
        prev = min(prev, candidate)
        adj[i] = min(prev, 1.0)
    restored = np.empty(m, dtype=float)
    restored[order] = adj
    out[mask] = restored
    return out


def write_excel_multisheet(path, sheet_map):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        for sheet_name, obj in sheet_map.items():
            if isinstance(obj, pd.DataFrame):
                obj.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            else:
                pd.DataFrame(obj).to_excel(
                    writer, sheet_name=sheet_name[:31], index=False
                )
    return path


def _resolve_output_dir(out_dir=None):
    if out_dir is not None:
        out_dir = Path(out_dir)
    elif "OUTPUT_DIR" in globals():
        out_dir = Path(OUTPUT_DIR)
    else:
        out_dir = Path.cwd() / "Conflict_Analysis_Outputs"

    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "figures").mkdir(parents=True, exist_ok=True)
    return out_dir


def save_dataframe(df, stem, out_dir=None):
    out_dir = _resolve_output_dir(out_dir)
    csv_path = out_dir / f"{stem}.csv"
    xlsx_path = out_dir / f"{stem}.xlsx"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=False, sheet_name="Sheet1")
    return csv_path, xlsx_path


def figure_path(name, out_dir=None):
    out_dir = _resolve_output_dir(out_dir)
    return out_dir / "figures" / name

## 3. Locate the uploaded benchmark workbook and the testing-sample files

This section finds:

- the **uploaded results workbook** from Colab content
- the **Testing Sample** folder on Google Drive
- the output folder for all generated tables and figures

In [ ]:
# --- UPLOADED-WORKBOOK RESOLVER: USE COLAB CONTENT FOR THE RESULTS SHEET ---

PROJECT_ROOT = find_existing_project_root()

testing_dir_candidates = [
    PROJECT_ROOT / "Assignments Part" / "Assignments Set" / "Testing Sample",
    PROJECT_ROOT / "Assignments Part" / "Assignments Set" / "Testing sample",
    PROJECT_ROOT / "Assignments Part" / "Assignments Set" / "testing sample",
]

TESTING_SAMPLE_DIR = None
for cand in testing_dir_candidates:
    if cand.exists():
        TESTING_SAMPLE_DIR = cand
        break

if TESTING_SAMPLE_DIR is None or not TESTING_SAMPLE_DIR.exists():
    raise FileNotFoundError(
        "Could not find the Testing Sample folder on Google Drive. "
        f"Checked under: {PROJECT_ROOT / 'Assignments Part' / 'Assignments Set'}"
    )

OUTPUT_DIR = OUTPUT_BASE_DIR / "Conflict_Analysis_Outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)


def uploaded_name_score(path_obj):
    name = path_obj.name.lower()
    score = 0
    for hint in UPLOADED_RESULTS_FILENAME_HINTS:
        hint_low = hint.lower()
        if hint_low in name:
            score += 100
    if path_obj.suffix.lower() in {".xlsx", ".xlsm", ".xls"}:
        score += 20
    if "combined" in name:
        score -= 80
    if "review" in name:
        score -= 30
    return score


def inspect_uploaded_workbook(candidate_path):
    try:
        sheet_map = load_excel_sheets(candidate_path)
    except Exception:
        return None

    best = None
    for s, df_probe in sheet_map.items():
        if df_probe is None or len(df_probe.columns) == 0:
            continue

        df_probe = df_probe.copy()
        df_probe.columns = [str(c).strip() for c in df_probe.columns]
        cols = {norm_header(c) for c in df_probe.columns}
        if not {"language", "label", "file_name"}.issubset(cols):
            continue

        cmap = {norm_header(c): c for c in df_probe.columns}
        label_col = cmap["label"]

        tmp = df_probe.dropna(how="all").copy()
        n_rows = len(tmp)
        n_nonnull_labels = tmp[label_col].replace("", pd.NA).notna().sum()
        n_unique_labels = (
            tmp[label_col].replace("", pd.NA).dropna().astype(str).str.strip().nunique()
        )

        if n_nonnull_labels == 0 or n_unique_labels < 2:
            continue

        row_score = 0
        if 430 <= n_rows <= 470:
            row_score += 100  # prefer the full 450-file workbook
        elif 280 <= n_rows <= 320:
            row_score += 60  # allow the 300-file workbook
        elif 250 <= n_rows <= 500:
            row_score += 20
        else:
            continue

        info = {
            "workbook": candidate_path,
            "sheet": s,
            "rows": int(n_rows),
            "nonnull_labels": int(n_nonnull_labels),
            "unique_labels": int(n_unique_labels),
            "sheet_names": list(sheet_map.keys()),
            "row_score": row_score,
        }
        if best is None or info["row_score"] > best["row_score"]:
            best = info

    return best


# Search uploaded Colab content only
candidate_files = []
for root in UPLOAD_SEARCH_DIRS:
    if root.exists():
        for suffix in {".xlsx", ".xlsm", ".xls", ".csv"}:
            candidate_files.extend(recursive_find_files(root, suffixes={suffix}))

# Deduplicate and sort
_seen = set()
candidate_files = [
    p for p in candidate_files if not (str(p) in _seen or _seen.add(str(p)))
]
candidate_files = sorted(candidate_files, key=uploaded_name_score, reverse=True)

print("Uploaded workbook candidates found:")
for p in candidate_files[:20]:
    print(" -", p)

accepted = []
for p in candidate_files:
    info = inspect_uploaded_workbook(p)
    if info is not None:
        accepted.append((p, info))

RESULTS_WORKBOOK = None
RESULTS_MAIN_SHEET = None
accepted_info = None

if accepted:
    accepted = sorted(
        accepted,
        key=lambda t: (t[1]["row_score"], uploaded_name_score(t[0])),
        reverse=True,
    )
    RESULTS_WORKBOOK, accepted_info = accepted[0]
    RESULTS_MAIN_SHEET = accepted_info["sheet"]

if RESULTS_WORKBOOK is None:
    raise FileNotFoundError(
        "Could not find a valid uploaded testing-sample workbook in Colab content.\n\n"
        "Please upload the results workbook into the Colab session. The workbook must contain a sheet with:\n"
        "  - Language\n"
        "  - Label\n"
        "  - file_name\n"
        "and approximately 300 or 450 labeled rows."
    )

print("\nAccepted workbook:")
print("Using TESTING_SAMPLE_DIR:", TESTING_SAMPLE_DIR)
print("Using RESULTS_WORKBOOK:", RESULTS_WORKBOOK)
print("Using RESULTS_MAIN_SHEET:", RESULTS_MAIN_SHEET)
print("Accepted info:", accepted_info)

try:
    _sheets = load_excel_sheets(RESULTS_WORKBOOK)
    print("Workbook sheets:", list(_sheets.keys()))
except Exception as e:
    print("Could not inspect workbook sheets:", e)

In [ ]:
# Resolver already ran in the previous cell.
# This cell is intentionally kept as a no-op so later cell numbering stays stable.

print("Resolver ready.")

In [ ]:
# Load workbook sheets
sheets = load_excel_sheets(RESULTS_WORKBOOK)
print("Workbook sheets:", list(sheets.keys()))

if RESULTS_MAIN_SHEET not in sheets:
    raise ValueError(
        f"Expected main sheet '{RESULTS_MAIN_SHEET}' not found in workbook {RESULTS_WORKBOOK}"
    )

main_sheet_name = RESULTS_MAIN_SHEET
main_df = sheets[main_sheet_name].copy()
main_df.columns = [str(c).strip() for c in main_df.columns]
main_df = main_df.dropna(how="all").copy()

print("Main sheet:", main_sheet_name)
print("Main sheet shape:", main_df.shape)
print("Columns:", main_df.columns.tolist())
if "Language" in main_df.columns:
    print("Language counts:")
    print(main_df["Language"].value_counts(dropna=False))
if "Label" in main_df.columns:
    print("Label counts:")
    print(main_df["Label"].value_counts(dropna=False))
display(main_df.head())

In [ ]:
# =========================
# TOOL CONSTANTS
# =========================

ALL_TOOLS = ["gptzero", "pangram", "sapling", "isgen"]
TEXT_TOOLS = ["gptzero", "pangram", "sapling", "isgen"]  # text languages use all four
CODE_TOOLS = ["pangram"]  # coding uses Pangram only

## 4. Build the master analysis table

This section:

- standardizes score columns  
- rebuilds predictions if needed  
- creates conflict indicators  
- adds consensus / disagreement metadata  
- maps each row to the underlying file on disk

In [ ]:
# Build canonical analysis dataframe

df = main_df.copy()

lang_col = first_matching_col(df, ["Language"], required=True)
label_col = first_matching_col(df, ["Label"], required=True)
file_col = first_matching_col(df, ["file_name", "filename", "file"], required=True)
word_col = first_matching_col(df, ["word_count", "wordcount", "words"])

df = df.dropna(subset=[lang_col, label_col, file_col]).copy()
df["Language"] = df[lang_col].map(language_family)
df["Label"] = df[label_col].map(norm_text)
df["file_name"] = df[file_col].map(norm_text)
df["truth_class"] = df["Label"].map(truth_class_from_label)
df["word_count_reported"] = safe_num(df[word_col]) if word_col is not None else np.nan

# Locate detector-specific fields while accommodating source-column variants.
tool_col_info = {}
for tool in ALL_TOOLS:
    rate_col = first_matching_col(
        df,
        [f"{tool}_ai_rate_percent", f"{tool}_score", f"{tool}_ai_score_percent"],
    )
    result_col = first_matching_col(df, [f"{tool}_result", f"{tool}_main_result"])
    pred_col = first_matching_col(df, [f"{tool}_pred_label_50"])
    soft_col = first_matching_col(df, [f"{tool}_soft_correctness_percent"])

    tool_col_info[tool] = {
        "rate_col": rate_col,
        "result_col": result_col,
        "pred_col": pred_col,
        "soft_col": soft_col,
        "scale": (
            detect_score_scale(df[rate_col])
            if rate_col is not None
            else "percent_0_100"
        ),
    }

for tool in ALL_TOOLS:
    info = tool_col_info[tool]

    if info["rate_col"] is not None:
        df[f"{tool}_score_percent"] = df[info["rate_col"]].apply(
            lambda value: to_score_percent(value, info["scale"])
        )
    else:
        df[f"{tool}_score_percent"] = np.nan

    if info["result_col"] is not None:
        df[f"{tool}_result_std"] = df[info["result_col"]].map(norm_text)
    else:
        df[f"{tool}_result_std"] = ""

    if info["pred_col"] is not None:
        df[f"{tool}_pred_label"] = df[info["pred_col"]].map(norm_text)
    else:
        df[f"{tool}_pred_label"] = df.apply(
            lambda row: binary_prediction_from_score_or_text(
                row[f"{tool}_score_percent"],
                row[f"{tool}_result_std"],
            ),
            axis=1,
        )

    if info["soft_col"] is not None:
        df[f"{tool}_soft_correctness"] = safe_num(df[info["soft_col"]])
    else:
        df[f"{tool}_soft_correctness"] = df.apply(
            lambda row: soft_correctness_percent(
                row[f"{tool}_score_percent"],
                row["truth_class"],
            ),
            axis=1,
        )

# Configure TESTING_SAMPLE_DIR outside the notebook for the private corpus location.
RELATIVE_DIR_MAP = {
    ("Arabic", "AI-Free"): Path("Arabic Assignments") / "Trimmed AI-Free Assignments",
    ("Arabic", "AI-Generated"): Path("Arabic Assignments") / "AI-Generated Assignments",
    ("Arabic", "Humanized AI"): Path("Arabic Assignments") / "Humanized AI Assignments",
    ("English", "AI-Free"): Path("English Assignments") / "Trimmed AI-Free Assignments",
    ("English", "AI-Generated"): Path("English Assignments")
    / "AI-Generated Assignments",
    ("English", "Humanized AI"): Path("English Assignments")
    / "Humanized AI Assignments",
    ("Coding", "AI-Free"): Path("Coding Assignments") / "AI-Free Code",
    ("Coding", "AI-Generated"): Path("Coding Assignments") / "AI-Generated Code",
    ("Coding", "Humanized AI"): Path("Coding Assignments") / "Humanized AI Code",
}


def build_file_path(row):
    relative_dir = RELATIVE_DIR_MAP.get((row["Language"], row["Label"]))
    if relative_dir is None:
        return np.nan

    path = TESTING_SAMPLE_DIR / relative_dir / row["file_name"]
    return str(path) if path.exists() else np.nan


df["file_path"] = df.apply(build_file_path, axis=1)
df["file_exists"] = df["file_path"].map(
    lambda path: Path(path).exists() if isinstance(path, str) and path else False
)

missing_files = df.loc[
    ~df["file_exists"],
    ["Language", "Label", "file_name", "file_path"],
]
print("Rows:", len(df))
print("Missing mapped files:", len(missing_files))
if len(missing_files):
    display(missing_files.head(20))

text_mask = df["Language"].isin(["Arabic", "English"])
coding_mask = df["Language"].eq("Coding")


def row_text_votes(row):
    votes = []
    for tool in TEXT_TOOLS:
        vote = row[f"{tool}_pred_label"]
        if pd.notna(vote) and norm_text(vote) != "":
            votes.append(vote)
    return votes


df["text_votes"] = df.apply(
    lambda row: row_text_votes(row) if row["Language"] in {"Arabic", "English"} else [],
    axis=1,
)
df["n_text_votes"] = df["text_votes"].map(len)
df["n_text_votes_ai"] = df["text_votes"].map(
    lambda votes: sum(vote == "AI" for vote in votes)
)
df["n_text_votes_human"] = df["text_votes"].map(
    lambda votes: sum(vote == "Human" for vote in votes)
)
df["is_text_conflict"] = df["text_votes"].map(
    lambda votes: len(set(votes)) > 1 if votes else False
)
df["is_text_unanimous"] = df["text_votes"].map(
    lambda votes: len(set(votes)) == 1 if votes else False
)
df["majority_vote_label"] = df.apply(
    lambda row: (
        "AI"
        if row["n_text_votes_ai"] > row["n_text_votes_human"]
        else "Human" if row["n_text_votes_human"] > row["n_text_votes_ai"] else "Tie"
    ),
    axis=1,
)
df["majority_matches_truth"] = np.where(
    text_mask & df["majority_vote_label"].isin(["AI", "Human"]),
    df["majority_vote_label"] == df["truth_class"],
    np.nan,
)

# Coding samples use Pangram as the available detector.
df["coding_pangram_correct"] = np.where(
    coding_mask,
    df["pangram_pred_label"] == df["truth_class"],
    np.nan,
)

score_columns = [f"{tool}_score_percent" for tool in TEXT_TOOLS]
score_values = df[score_columns].astype(float).to_numpy()
for aggregate_name, function in {
    "mean": np.nanmean,
    "std": np.nanstd,
    "min": np.nanmin,
    "max": np.nanmax,
}.items():
    with np.errstate(all="ignore"):
        values = np.array(
            [
                function(row) if np.isfinite(row).any() else np.nan
                for row in score_values
            ],
            dtype=float,
        )
    df[f"text_score_{aggregate_name}"] = np.where(text_mask, values, np.nan)

df["text_score_range"] = np.where(
    text_mask,
    df["text_score_max"] - df["text_score_min"],
    np.nan,
)
df["text_score_margin_from_50_abs"] = np.where(
    text_mask,
    (df["text_score_mean"] - 50.0).abs(),
    np.nan,
)


def pattern_string(row):
    if row["Language"] not in {"Arabic", "English"}:
        return ""
    return " | ".join(f"{tool}:{row[f'{tool}_pred_label']}" for tool in TEXT_TOOLS)


df["vote_pattern"] = df.apply(pattern_string, axis=1)

master_csv, master_xlsx = save_dataframe(df, "analysis_master_table")
print("Saved master table to:")
print(master_csv)
print(master_xlsx)

display(df.head())

## 5. Dataset audit

This section confirms that the benchmark is balanced and that the file mapping succeeded.

In [ ]:
dataset_audit = (
    df.groupby(["Language", "Label"], dropna=False)
    .agg(
        N=("file_name", "size"),
        Files_Found=("file_exists", "sum"),
        Mean_Reported_Words=("word_count_reported", "mean"),
    )
    .reset_index()
)

display(dataset_audit)

audit_csv, audit_xlsx = save_dataframe(dataset_audit, "dataset_audit")
print(audit_csv)

## 6. Core conflict counts

Definitions used in this notebook:

- **Arabic / English conflict**: the four text tools do **not all agree** on the binary label at the 50 threshold  
- **Coding conflict proxy**: since only Pangram exists, the notebook uses **correct vs incorrect relative to ground truth** instead of tool-vs-tool disagreement

In [ ]:
text_conflict_counts = (
    df[text_mask]
    .groupby(["Language", "Label"], dropna=False)
    .agg(
        N=("file_name", "size"),
        Conflicting=("is_text_conflict", "sum"),
        Unanimous=("is_text_unanimous", "sum"),
    )
    .reset_index()
)
text_conflict_counts["Conflict_Rate_Percent"] = (
    100 * text_conflict_counts["Conflicting"] / text_conflict_counts["N"]
).round(2)
text_conflict_counts["Unanimous_Rate_Percent"] = (
    100 * text_conflict_counts["Unanimous"] / text_conflict_counts["N"]
).round(2)

coding_accuracy_counts = (
    df[coding_mask]
    .groupby(["Language", "Label"], dropna=False)
    .agg(
        N=("file_name", "size"),
        Correct=("coding_pangram_correct", lambda s: pd.Series(s).eq(True).sum()),
        Incorrect=("coding_pangram_correct", lambda s: pd.Series(s).eq(False).sum()),
    )
    .reset_index()
)
coding_accuracy_counts["Accuracy_Percent"] = (
    100 * coding_accuracy_counts["Correct"] / coding_accuracy_counts["N"]
).round(2)

display(text_conflict_counts)
display(coding_accuracy_counts)

save_dataframe(text_conflict_counts, "text_conflict_counts_by_language_label")
save_dataframe(coding_accuracy_counts, "coding_accuracy_counts_by_label")

In [ ]:
# Plot: text conflict rates
plot_df = text_conflict_counts.copy()
langs = ["Arabic", "English"]
labels = ["AI-Free", "AI-Generated", "Humanized AI"]
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
for i, lang in enumerate(langs):
    vals = [
        plot_df.loc[
            (plot_df["Language"] == lang) & (plot_df["Label"] == lab),
            "Conflict_Rate_Percent",
        ].iloc[0]
        for lab in labels
    ]
    ax.bar(x + (i - 0.5) * width, vals, width=width, label=lang)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=0)
ax.set_ylabel("Conflict rate (%)")
ax.set_title("Text-tool conflict rate by language and label")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

fp = figure_path("text_conflict_rates_by_language_label.png")
plt.savefig(fp, dpi=220, bbox_inches="tight")
plt.show()
print(fp)

## 7. Tool-disagreement structure

This section identifies:

- dominant vote patterns  
- lone dissenters  
- 2-vs-2 splits  
- unanimous AI vs unanimous Human outcomes  
- how these patterns vary by language and label

In [ ]:
def classify_pattern(row):
    if row["Language"] not in {"Arabic", "English"}:
        return pd.Series(
            {
                "pattern_type": "",
                "lone_dissenter_tool": "",
                "pair_ai": "",
                "pair_human": "",
                "unanimous_label": "",
            }
        )
    votes = {tool: row[f"{tool}_pred_label"] for tool in TEXT_TOOLS}
    vote_counts = Counter(votes.values())
    unique_labels = [k for k in vote_counts if pd.notna(k) and norm_text(k) != ""]
    if len(unique_labels) == 1:
        unanimous_label = unique_labels[0]
        return pd.Series(
            {
                "pattern_type": "Unanimous",
                "lone_dissenter_tool": "",
                "pair_ai": "",
                "pair_human": "",
                "unanimous_label": unanimous_label,
            }
        )
    if sorted(vote_counts.values()) == [1, 3]:
        minority_label = min(vote_counts, key=lambda k: vote_counts[k])
        lone_tool = [tool for tool, lbl in votes.items() if lbl == minority_label][0]
        return pd.Series(
            {
                "pattern_type": "Lone dissenter",
                "lone_dissenter_tool": lone_tool,
                "pair_ai": "",
                "pair_human": "",
                "unanimous_label": "",
            }
        )
    if sorted(vote_counts.values()) == [2, 2]:
        ai_pair = [tool for tool, lbl in votes.items() if lbl == "AI"]
        human_pair = [tool for tool, lbl in votes.items() if lbl == "Human"]
        return pd.Series(
            {
                "pattern_type": "2 vs 2 split",
                "lone_dissenter_tool": "",
                "pair_ai": " + ".join(ai_pair),
                "pair_human": " + ".join(human_pair),
                "unanimous_label": "",
            }
        )
    return pd.Series(
        {
            "pattern_type": "Other",
            "lone_dissenter_tool": "",
            "pair_ai": "",
            "pair_human": "",
            "unanimous_label": "",
        }
    )


pattern_meta = df.apply(classify_pattern, axis=1)
for c in pattern_meta.columns:
    df[c] = pattern_meta[c]

pattern_counts = (
    df[text_mask]
    .groupby(["Language", "Label", "pattern_type"], dropna=False)
    .size()
    .reset_index(name="N")
    .sort_values(["Language", "Label", "N"], ascending=[True, True, False])
)

lone_dissenter_counts = (
    df[text_mask & df["pattern_type"].eq("Lone dissenter")]
    .groupby(["Language", "Label", "lone_dissenter_tool"], dropna=False)
    .size()
    .reset_index(name="N")
    .sort_values(["Language", "Label", "N"], ascending=[True, True, False])
)

split_counts = (
    df[text_mask & df["pattern_type"].eq("2 vs 2 split")]
    .assign(split_signature=lambda x: x["pair_ai"] + "  vs  " + x["pair_human"])
    .groupby(["Language", "Label", "split_signature"], dropna=False)
    .size()
    .reset_index(name="N")
    .sort_values(["Language", "Label", "N"], ascending=[True, True, False])
)

display(pattern_counts.head(50))
display(lone_dissenter_counts.head(50))
display(split_counts.head(50))

save_dataframe(pattern_counts, "pattern_counts")
save_dataframe(lone_dissenter_counts, "lone_dissenter_counts")
save_dataframe(split_counts, "two_vs_two_split_counts")

In [ ]:
# Plot top split signatures for humanized subsets
split_plot_df = split_counts[split_counts["Label"].eq("Humanized AI")].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)
for ax, lang in zip(axes, ["Arabic", "English"]):
    sub = split_plot_df[split_plot_df["Language"].eq(lang)].head(8)
    ax.barh(sub["split_signature"], sub["N"])
    ax.set_title(f"{lang} humanized: top 2-vs-2 split patterns")
    ax.set_xlabel("Count")
    ax.invert_yaxis()
plt.tight_layout()

fp = figure_path("humanized_split_patterns.png")
plt.savefig(fp, dpi=220, bbox_inches="tight")
plt.show()
print(fp)

## 8. Tool-level conflict and error contribution

This section quantifies which tool is driving conflict most often, and in which subsets.

In [ ]:
tool_subset_rows = []
for lang in ["Arabic", "English"]:
    for label in ["AI-Free", "AI-Generated", "Humanized AI"]:
        sub = df[(df["Language"] == lang) & (df["Label"] == label)].copy()
        for tool in TEXT_TOOLS:
            pred = sub[f"{tool}_pred_label"]
            truth = sub["truth_class"]
            n = len(sub)
            correct = (pred == truth).sum()
            fp = ((pred == "AI") & (truth == "Human")).sum()
            fn = ((pred == "Human") & (truth == "AI")).sum()
            lone_diss = (
                (sub["pattern_type"] == "Lone dissenter")
                & (sub["lone_dissenter_tool"] == tool)
            ).sum()
            tool_subset_rows.append(
                {
                    "Language": lang,
                    "Label": label,
                    "Tool": tool,
                    "N": n,
                    "Correct": correct,
                    "Accuracy_Percent": round(100 * correct / n, 4) if n else np.nan,
                    "False_Positives": fp,
                    "False_Negatives": fn,
                    "Lone_Dissenter_Count": lone_diss,
                }
            )

tool_subset_df = pd.DataFrame(tool_subset_rows)
display(tool_subset_df.head(50))
save_dataframe(tool_subset_df, "tool_level_subset_accuracy_and_dissent")

## 9. Score-spread and uncertainty analysis

This section studies whether conflicting files also show larger cross-tool score spread.

In [ ]:
uncertainty_summary = (
    df[text_mask]
    .groupby(["Language", "Label", "is_text_conflict"], dropna=False)
    .agg(
        N=("file_name", "size"),
        Mean_Tool_Score_Mean=("text_score_mean", "mean"),
        Mean_Tool_Score_STD=("text_score_std", "mean"),
        Mean_Tool_Score_Range=("text_score_range", "mean"),
        Median_Tool_Score_Range=("text_score_range", "median"),
        Mean_Abs_Margin_From_50=("text_score_margin_from_50_abs", "mean"),
    )
    .reset_index()
)
display(uncertainty_summary)
save_dataframe(uncertainty_summary, "uncertainty_summary")

In [ ]:
# Compare score spread in conflict vs non-conflict for text subsets
unc_rows = []
for lang in ["Arabic", "English"]:
    for label in ["AI-Free", "AI-Generated", "Humanized AI"]:
        sub = df[(df["Language"] == lang) & (df["Label"] == label)].copy()
        a = sub.loc[sub["is_text_conflict"], "text_score_range"].dropna()
        b = sub.loc[~sub["is_text_conflict"], "text_score_range"].dropna()
        p = np.nan
        if len(a) >= 2 and len(b) >= 2:
            p = mannwhitneyu(a, b, alternative="two-sided").pvalue
        unc_rows.append(
            {
                "Language": lang,
                "Label": label,
                "Conflict_N": len(a),
                "NonConflict_N": len(b),
                "Conflict_Median_Score_Range": a.median() if len(a) else np.nan,
                "NonConflict_Median_Score_Range": b.median() if len(b) else np.nan,
                "Cliffs_Delta": cliffs_delta(a, b),
                "P_Value": p,
            }
        )

unc_test_df = pd.DataFrame(unc_rows)
unc_test_df["BH_Q_Value"] = benjamini_hochberg(unc_test_df["P_Value"].tolist())
display(unc_test_df)
save_dataframe(unc_test_df, "uncertainty_conflict_vs_nonconflict_tests")

## 10. Extract text/code features from every file

This is the main content-analysis section.  
It builds a feature table for every file, then merges it with the detector results.

The feature set is intentionally broad and includes:

- length and density measures  
- lexical diversity  
- punctuation and digit density  
- headings / bullets / repeated lines  
- language-specific word statistics  
- code-style structural features for the coding sample

In [ ]:
# =========================
# FEATURE EXTRACTION
# =========================

AR_WORD_RE = re.compile(r"[\u0600-\u06FF]+")
EN_WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
CODE_TOKEN_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
NUMBER_RE = re.compile(r"\d")
PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)


def read_text_file(path):
    path = Path(path)
    if not path.exists():
        return ""
    # Try utf-8 first, then safe fallbacks
    for enc in ["utf-8", "utf-8-sig", "cp1256", "cp1252", "latin-1"]:
        try:
            return path.read_text(encoding=enc)
        except Exception:
            pass
    # binary fallback
    with open(path, "rb") as f:
        raw = f.read()
    try:
        import chardet

        enc = chardet.detect(raw).get("encoding") or "utf-8"
        return raw.decode(enc, errors="replace")
    except Exception:
        return raw.decode("utf-8", errors="replace")


def shannon_entropy(text):
    if not text:
        return 0.0
    counts = Counter(text)
    n = len(text)
    return -sum((v / n) * math.log2(v / n) for v in counts.values())


def text_tokens(text, language):
    if language == "Arabic":
        return AR_WORD_RE.findall(text)
    if language == "English":
        return EN_WORD_RE.findall(text)
    return CODE_TOKEN_RE.findall(text)


def heading_like(line):
    s = line.strip()
    if not s:
        return False
    if len(s) < 120 and re.match(r"^\d+(\.\d+)*[\)\.]?\s+\S", s):
        return True
    if re.match(
        r"^(chapter|section|abstract|introduction|method|methodology|results?|discussion|conclusion|references)\b",
        s,
        re.I,
    ):
        return True
    if re.match(r"^[A-Z][A-Z \-]{4,}$", s):
        return True
    if re.match(r"^[\u0600-\u06FF\s]{2,}[:：]?$", s):
        return True
    return False


def bullet_like(line):
    s = line.strip()
    if not s:
        return False
    if re.match(r"^([-*•])\s+", s):
        return True
    if re.match(r"^\d+[\.)]\s+", s):
        return True
    if re.match(r"^[A-Za-z][\.)]\s+", s):
        return True
    return False


def code_specific_features(text):
    lines = text.splitlines()
    nonblank = [ln for ln in lines if ln.strip()]
    return {
        "code_import_count": len(
            re.findall(r"^\s*(import\s+|from\s+.+\s+import\s+)", text, flags=re.M)
        ),
        "code_comment_line_count": len(
            re.findall(r"^\s*(#|//|/\*|\*)", text, flags=re.M)
        ),
        "code_function_like_count": len(
            re.findall(
                r"\b(def|function|func|void|int|float|double|public|private|protected|static)\b",
                text,
            )
        ),
        "code_class_like_count": len(re.findall(r"\bclass\b", text)),
        "code_brace_count": text.count("{") + text.count("}"),
        "code_semicolon_count": text.count(";"),
        "code_indent_line_count": sum(
            1 for ln in nonblank if re.match(r"^\s{2,}\S", ln)
        ),
        "code_blank_line_ratio": (len(lines) - len(nonblank)) / max(len(lines), 1),
    }


def extract_features_from_text(text, language):
    lines = text.splitlines()
    nonblank = [ln for ln in lines if ln.strip()]
    blank_line_count = len(lines) - len(nonblank)
    paragraphs = [p for p in re.split(r"\n\s*\n+", text) if p.strip()]

    toks = text_tokens(text, language)
    toks_lower = [t.lower() for t in toks]
    uniq = len(set(toks_lower))

    avg_para_words = np.nan
    if paragraphs:
        para_lengths = [len(text_tokens(p, language)) for p in paragraphs]
        avg_para_words = float(np.mean(para_lengths)) if len(para_lengths) else np.nan

    repeated_lines = Counter(ln.strip() for ln in nonblank if len(ln.strip()) >= 5)
    repeated_line_count = sum(v - 1 for v in repeated_lines.values() if v > 1)

    feat = {
        "char_count": len(text),
        "line_count": len(lines),
        "nonblank_line_count": len(nonblank),
        "blank_line_count": blank_line_count,
        "paragraph_count": len(paragraphs),
        "word_count_extracted": len(toks),
        "unique_word_count": uniq,
        "type_token_ratio": uniq / len(toks) if len(toks) else np.nan,
        "avg_word_length": (
            float(np.mean([len(t) for t in toks])) if len(toks) else np.nan
        ),
        "avg_line_length": (
            float(np.mean([len(ln) for ln in nonblank])) if len(nonblank) else np.nan
        ),
        "avg_paragraph_words": avg_para_words,
        "digit_count": len(NUMBER_RE.findall(text)),
        "digit_density": len(NUMBER_RE.findall(text)) / max(len(text), 1),
        "punctuation_count": len(PUNCT_RE.findall(text)),
        "punctuation_density": len(PUNCT_RE.findall(text)) / max(len(text), 1),
        "heading_like_line_count": sum(1 for ln in nonblank if heading_like(ln)),
        "heading_density": sum(1 for ln in nonblank if heading_like(ln))
        / max(len(nonblank), 1),
        "bullet_like_line_count": sum(1 for ln in nonblank if bullet_like(ln)),
        "bullet_density": sum(1 for ln in nonblank if bullet_like(ln))
        / max(len(nonblank), 1),
        "repeated_line_count": repeated_line_count,
        "repeated_line_density": repeated_line_count / max(len(nonblank), 1),
        "char_entropy": shannon_entropy(text),
        "arabic_char_ratio": len(re.findall(r"[\u0600-\u06FF]", text))
        / max(len(text), 1),
        "latin_char_ratio": len(re.findall(r"[A-Za-z]", text)) / max(len(text), 1),
    }

    if language == "Coding":
        feat.update(code_specific_features(text))

    return feat


# Cache features to avoid recomputation on rerun
features_cache_path = OUTPUT_DIR / "features_cache.parquet"

if features_cache_path.exists():
    features_df = pd.read_parquet(features_cache_path)
    print("Loaded feature cache:", features_cache_path)
else:
    feature_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
        path = row["file_path"]
        text = read_text_file(path) if isinstance(path, str) and path else ""
        feats = extract_features_from_text(
            text,
            row["Language"] if row["Language"] in {"Arabic", "English"} else "Coding",
        )
        out = {
            "Language": row["Language"],
            "Label": row["Label"],
            "file_name": row["file_name"],
            "file_path": row["file_path"],
            "raw_text_length_chars": len(text),
        }
        out.update(feats)
        feature_rows.append(out)

    features_df = pd.DataFrame(feature_rows)
    features_df.to_parquet(features_cache_path, index=False)
    print("Saved feature cache:", features_cache_path)

display(features_df.head())

In [ ]:
# Merge features into analysis master
merged = df.merge(
    features_df,
    on=["Language", "Label", "file_name", "file_path"],
    how="left",
    validate="one_to_one",
)

merged_csv, merged_xlsx = save_dataframe(merged, "analysis_master_with_features")
print(merged_csv)
print(merged_xlsx)
print("Merged shape:", merged.shape)

## 11. Numeric feature-comparison engine

This section runs the key comparisons discussed earlier:

- Arabic AI-Free: conflict vs non-conflict  
- Arabic AI-Generated: conflict vs non-conflict  
- Arabic Humanized AI: conflict vs non-conflict  
- English AI-Free: conflict vs non-conflict  
- English Humanized AI: conflict vs non-conflict  
- Coding AI-generated + Humanized: correct vs incorrect

In [ ]:
NUMERIC_FEATURES = [
    "word_count_reported",
    "word_count_extracted",
    "char_count",
    "line_count",
    "nonblank_line_count",
    "blank_line_count",
    "paragraph_count",
    "unique_word_count",
    "type_token_ratio",
    "avg_word_length",
    "avg_line_length",
    "avg_paragraph_words",
    "digit_count",
    "digit_density",
    "punctuation_count",
    "punctuation_density",
    "heading_like_line_count",
    "heading_density",
    "bullet_like_line_count",
    "bullet_density",
    "repeated_line_count",
    "repeated_line_density",
    "char_entropy",
    "text_score_mean",
    "text_score_std",
    "text_score_min",
    "text_score_max",
    "text_score_range",
    "text_score_margin_from_50_abs",
    "arabic_char_ratio",
    "latin_char_ratio",
    "code_import_count",
    "code_comment_line_count",
    "code_function_like_count",
    "code_class_like_count",
    "code_brace_count",
    "code_semicolon_count",
    "code_indent_line_count",
    "code_blank_line_ratio",
]


def compare_binary_groups(data, mask, group_col, feature_cols):
    sub = data[mask].copy()
    a = sub[sub[group_col] == True].copy()
    b = sub[sub[group_col] == False].copy()
    rows = []
    for feat in feature_cols:
        x = pd.to_numeric(a[feat], errors="coerce").dropna()
        y = pd.to_numeric(b[feat], errors="coerce").dropna()
        p = np.nan
        if len(x) >= 2 and len(y) >= 2:
            try:
                p = mannwhitneyu(x, y, alternative="two-sided").pvalue
            except Exception:
                p = np.nan
        rows.append(
            {
                "feature": feat,
                "group1_n": len(x),
                "group0_n": len(y),
                "group1_median": x.median() if len(x) else np.nan,
                "group0_median": y.median() if len(y) else np.nan,
                "group1_mean": x.mean() if len(x) else np.nan,
                "group0_mean": y.mean() if len(y) else np.nan,
                "delta_mean": (x.mean() - y.mean()) if len(x) and len(y) else np.nan,
                "cliffs_delta": cliffs_delta(x, y),
                "p_value": p,
            }
        )
    out = pd.DataFrame(rows)
    out["bh_q_value"] = benjamini_hochberg(out["p_value"].tolist())
    out["abs_cliffs_delta"] = out["cliffs_delta"].abs()
    out = out.sort_values(
        ["bh_q_value", "p_value", "abs_cliffs_delta"], ascending=[True, True, False]
    )
    return out


comparison_specs = {
    "arabic_ai_free_conflict_vs_nonconflict": {
        "mask": (merged["Language"] == "Arabic") & (merged["Label"] == "AI-Free"),
        "group_col": "is_text_conflict",
    },
    "arabic_ai_generated_conflict_vs_nonconflict": {
        "mask": (merged["Language"] == "Arabic") & (merged["Label"] == "AI-Generated"),
        "group_col": "is_text_conflict",
    },
    "arabic_humanized_conflict_vs_nonconflict": {
        "mask": (merged["Language"] == "Arabic") & (merged["Label"] == "Humanized AI"),
        "group_col": "is_text_conflict",
    },
    "english_ai_free_conflict_vs_nonconflict": {
        "mask": (merged["Language"] == "English") & (merged["Label"] == "AI-Free"),
        "group_col": "is_text_conflict",
    },
    "english_humanized_conflict_vs_nonconflict": {
        "mask": (merged["Language"] == "English") & (merged["Label"] == "Humanized AI"),
        "group_col": "is_text_conflict",
    },
    "coding_ai_and_humanized_correct_vs_incorrect": {
        "mask": (merged["Language"] == "Coding")
        & (merged["Label"].isin(["AI-Generated", "Humanized AI"])),
        "group_col": "coding_pangram_correct",
    },
}

comparison_results = {}
for name, spec in comparison_specs.items():
    comp = compare_binary_groups(
        merged, spec["mask"], spec["group_col"], NUMERIC_FEATURES
    )
    comparison_results[name] = comp
    save_dataframe(comp, name)

for name, comp in comparison_results.items():
    print("\n", "=" * 100)
    print(name)
    display(comp.head(15))

## 12. High-yield exemplars

This section saves concrete file lists for follow-up reading.

It produces:

- top conflicting files by score spread  
- top non-conflicting files by score certainty  
- coding false negatives  
- coding true negatives  
- representative files per dominant split pattern

In [ ]:
# Top text conflicts by score spread

# Rebuild masks on the SAME dataframe
text_mask_merged = merged["Language"].isin(["Arabic", "English"])
coding_mask_merged = merged["Language"].eq("Coding")

text_exemplars = merged.loc[text_mask_merged].copy()

top_conflict_spread = text_exemplars.loc[
    text_exemplars["is_text_conflict"]
].sort_values(
    ["Language", "Label", "text_score_range", "text_score_std"],
    ascending=[True, True, False, False],
)[
    [
        "Language",
        "Label",
        "file_name",
        "file_path",
        "text_score_mean",
        "text_score_std",
        "text_score_range",
        "vote_pattern",
    ]
    + [f"{t}_score_percent" for t in TEXT_TOOLS]
    + [f"{t}_pred_label" for t in TEXT_TOOLS]
]

top_nonconflict_certainty = text_exemplars.loc[
    ~text_exemplars["is_text_conflict"]
].sort_values(
    ["Language", "Label", "text_score_range", "text_score_std"],
    ascending=[True, True, True, True],
)[
    [
        "Language",
        "Label",
        "file_name",
        "file_path",
        "text_score_mean",
        "text_score_std",
        "text_score_range",
        "vote_pattern",
    ]
    + [f"{t}_score_percent" for t in TEXT_TOOLS]
    + [f"{t}_pred_label" for t in TEXT_TOOLS]
]

coding_false_negatives = merged.loc[
    coding_mask_merged
    & (merged["truth_class"] == "AI")
    & (merged["pangram_pred_label"] == "Human")
].sort_values(["Label", "pangram_score_percent"], ascending=[True, True])[
    [
        "Language",
        "Label",
        "file_name",
        "file_path",
        "pangram_score_percent",
        "pangram_pred_label",
        "word_count_extracted",
        "code_import_count",
        "code_comment_line_count",
        "code_function_like_count",
        "code_class_like_count",
        "code_blank_line_ratio",
        "repeated_line_count",
    ]
]

coding_true_negatives = merged.loc[
    coding_mask_merged
    & (merged["truth_class"] == "Human")
    & (merged["pangram_pred_label"] == "Human")
].sort_values(["pangram_score_percent"], ascending=[True])[
    [
        "Language",
        "Label",
        "file_name",
        "file_path",
        "pangram_score_percent",
        "pangram_pred_label",
        "word_count_extracted",
        "code_import_count",
        "code_comment_line_count",
        "code_function_like_count",
        "code_class_like_count",
        "code_blank_line_ratio",
        "repeated_line_count",
    ]
]

save_dataframe(top_conflict_spread, "top_conflict_spread_exemplars")
save_dataframe(top_nonconflict_certainty, "top_nonconflict_certainty_exemplars")
save_dataframe(coding_false_negatives, "coding_false_negatives")
save_dataframe(coding_true_negatives, "coding_true_negatives")

display(top_conflict_spread.head(20))
display(coding_false_negatives.head(20))

## 13. Summary narratives generated from the numeric results

This section produces **plain-language, data-grounded summaries** that can be used later when writing the paper.

In [ ]:
summary_lines = []

# Core conflict statements
for _, r in text_conflict_counts.iterrows():
    summary_lines.append(
        f"{r['Language']} {r['Label']}: {int(r['Conflicting'])}/{int(r['N'])} conflicting "
        f"({r['Conflict_Rate_Percent']:.2f}%)."
    )

for _, r in coding_accuracy_counts.iterrows():
    summary_lines.append(
        f"Coding {r['Label']}: Pangram correct on {int(r['Correct'])}/{int(r['N'])} files "
        f"({r['Accuracy_Percent']:.2f}%)."
    )

# Top lone dissenters
top_lone = lone_dissenter_counts.groupby(["Language", "Label"], as_index=False).first()
for _, r in top_lone.iterrows():
    summary_lines.append(
        f"In {r['Language']} {r['Label']}, the most frequent lone dissenter was {r['lone_dissenter_tool']} "
        f"(N={int(r['N'])})."
    )

# Top 2-vs-2 splits
top_splits = split_counts.groupby(["Language", "Label"], as_index=False).first()
for _, r in top_splits.iterrows():
    summary_lines.append(
        f"In {r['Language']} {r['Label']}, the most frequent 2-vs-2 split was '{r['split_signature']}' "
        f"(N={int(r['N'])})."
    )

summary_text = "\n".join(f"- {line}" for line in summary_lines)
print(summary_text)

summary_md_path = OUTPUT_DIR / "conflict_analysis_summary.md"
summary_md_path.write_text(
    "# Conflict Analysis Summary\n\n" + summary_text, encoding="utf-8"
)
print(summary_md_path)

## 14. Optional OpenAI-based qualitative review

This section is **off by default**.  
Enable it only if you want model-assisted qualitative comparisons between conflicting and non-conflicting files.

It can help with pattern interpretation, but the notebook's **official numeric results do not depend on it**.

In [ ]:
import os
import getpass

if USE_OPENAI_QUAL_REVIEW:
    os.environ[OPENAI_API_KEY_ENV] = getpass.getpass("Enter OpenAI API key: ")
    print(f"{OPENAI_API_KEY_ENV} is set.")
else:
    print("USE_OPENAI_QUAL_REVIEW is False.")

In [ ]:
import os

if USE_OPENAI_QUAL_REVIEW:
    try:
        from openai import OpenAI

        api_key = os.environ.get(OPENAI_API_KEY_ENV, "").strip()
        if not api_key:
            raise RuntimeError(
                f"Environment variable {OPENAI_API_KEY_ENV} is not set. "
                "Set it first if you want OpenAI qualitative review."
            )
        client = OpenAI(api_key=api_key)
    except Exception as e:
        USE_OPENAI_QUAL_REVIEW = False
        print("OpenAI qualitative review disabled:", e)


def build_qual_review_sample(
    data, mask, n=None, sort_col="text_score_range", ascending=False
):
    sub = data[mask].copy()
    if len(sub) == 0:
        return sub
    if n is None:
        n = OPENAI_MAX_FILES_PER_SUBSET
    return sub.sort_values(sort_col, ascending=ascending).head(n)


def clip_text_for_model(path, max_chars=8000):
    txt = read_text_file(path)
    return txt[:max_chars]


if USE_OPENAI_QUAL_REVIEW:
    review_rows = []
    review_spec = [
        (
            "Arabic Humanized conflict",
            (merged["Language"] == "Arabic")
            & (merged["Label"] == "Humanized AI")
            & (merged["is_text_conflict"]),
        ),
        (
            "English Humanized conflict",
            (merged["Language"] == "English")
            & (merged["Label"] == "Humanized AI")
            & (merged["is_text_conflict"]),
        ),
        (
            "Arabic AI-Free conflict",
            (merged["Language"] == "Arabic")
            & (merged["Label"] == "AI-Free")
            & (merged["is_text_conflict"]),
        ),
        (
            "Coding AI false negatives",
            (merged["Language"] == "Coding")
            & (merged["truth_class"] == "AI")
            & (merged["pangram_pred_label"] == "Human"),
        ),
    ]

    for title, mask in review_spec:
        sample = build_qual_review_sample(merged, mask, n=OPENAI_MAX_FILES_PER_SUBSET)
        for _, row in sample.iterrows():
            prompt = f"""
You are reviewing a benchmark file from an AI-detection conflict study.

Task:
1. Briefly describe the dominant structural and stylistic patterns in this file.
2. Explain why such a file might cause conflict or false negatives for AI detectors.
3. Keep the answer factual and concise.
4. Do not speculate beyond the file content.

Metadata:
- Language: {row['Language']}
- Label: {row['Label']}
- File: {row['file_name']}

File content:
{clip_text_for_model(row['file_path'])}
"""
            try:
                resp = client.responses.create(
                    model=OPENAI_MODEL,
                    input=prompt,
                    temperature=0.2,
                )
                text = getattr(resp, "output_text", "")
            except Exception as e:
                text = f"[ERROR] {e}"

            review_rows.append(
                {
                    "Review_Group": title,
                    "Language": row["Language"],
                    "Label": row["Label"],
                    "file_name": row["file_name"],
                    "file_path": row["file_path"],
                    "Model_Review": text,
                }
            )

    qual_df = pd.DataFrame(review_rows)
    save_dataframe(qual_df, "openai_qualitative_reviews")
    display(qual_df)
else:
    print("USE_OPENAI_QUAL_REVIEW is False, so this section is skipped.")

In [ ]:
openai_csv = OUTPUT_DIR / "openai_qualitative_reviews.csv"
openai_xlsx = OUTPUT_DIR / "openai_qualitative_reviews.xlsx"

print("CSV exists:", openai_csv.exists(), openai_csv)
print("XLSX exists:", openai_xlsx.exists(), openai_xlsx)

if "qual_df" in globals():
    print("Rows:", len(qual_df))
    display(qual_df.head(10))

## 15. Final export workbook

This cell writes a single multi-sheet workbook containing the main results of this notebook.

In [ ]:
final_export_path = OUTPUT_DIR / "Testing_Sample_Conflict_Analysis_Full_Results.xlsx"

sheet_map = {
    "dataset_audit": dataset_audit,
    "analysis_master": df,
    "analysis_master_with_features": merged,
    "text_conflict_counts": text_conflict_counts,
    "coding_accuracy_counts": coding_accuracy_counts,
    "pattern_counts": pattern_counts,
    "lone_dissenter_counts": lone_dissenter_counts,
    "split_counts": split_counts,
    "tool_level_subset_accuracy": tool_subset_df,
    "uncertainty_summary": uncertainty_summary,
    "uncertainty_tests": unc_test_df,
    "top_conflict_spread": top_conflict_spread.head(300),
    "top_nonconflict_certainty": top_nonconflict_certainty.head(300),
    "coding_false_negatives": coding_false_negatives.head(300),
    "coding_true_negatives": coding_true_negatives.head(300),
}

for name, comp in comparison_results.items():
    sheet_map[name[:31]] = comp.head(500)

write_excel_multisheet(final_export_path, sheet_map)
print(final_export_path)

## 16. Quick interpretation checklist

After the notebook finishes, check these files first:

1. `Testing_Sample_Conflict_Analysis_Full_Results.xlsx`  
2. `text_conflict_counts_by_language_label.csv`  
3. `lone_dissenter_counts.csv`  
4. `two_vs_two_split_counts.csv`  
5. the feature-comparison tables, especially:
   - `arabic_ai_free_conflict_vs_nonconflict`
   - `arabic_humanized_conflict_vs_nonconflict`
   - `english_humanized_conflict_vs_nonconflict`
   - `coding_ai_and_humanized_correct_vs_incorrect`
6. the figures folder under `Conflict_Analysis_Outputs/figures`

These are the core outputs that should feed the paper's conflict-analysis subsection.

In [ ]:
print("All done.")
print("Results workbook:", RESULTS_WORKBOOK)
print("Testing sample folder:", TESTING_SAMPLE_DIR)
print("Output directory:", OUTPUT_DIR)

print("\nGenerated files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(p)

In [ ]:
import os
import re
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# Configure the directory containing analysis outputs before running.
PROJECT_DATA_PATH = os.environ.get("PROJECT_DATA_PATH")
MOUNT_GOOGLE_DRIVE = os.environ.get("MOUNT_GOOGLE_DRIVE", "false").lower() == "true"

if not PROJECT_DATA_PATH:
    raise ValueError(
        "Set the PROJECT_DATA_PATH environment variable to the output directory."
    )

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "Google Drive mounting is available only in Google Colab."
        ) from exc
    drive.mount("/content/drive")

OUTPUT_DIR = Path(PROJECT_DATA_PATH)
MASTER_WORKBOOK_NAME = "Conflict_Analysis_Outputs_Consolidated.xlsx"
BACKUP_ZIP_NAME = "Conflict_Analysis_Outputs_Backup.zip"
MOVE_ORIGINALS_TO_ARCHIVE_SUBFOLDER = False
ARCHIVE_SUBFOLDER_NAME = "_Archived_Original_Files"

PREFERRED_MASTER_STEMS = [
    "testing_sample_conflict_analysis_full_results",
    "testing_sample_conflict_analysis_full",
    "full_results",
]

IGNORE_FILE_STEMS = {
    "conflict_analysis_outputs_consolidated",
    "conflict_analysis_outputs_backup",
}
IGNORE_FILE_NAMES = {
    MASTER_WORKBOOK_NAME.lower(),
    BACKUP_ZIP_NAME.lower(),
}


def norm_text(value):
    return "" if value is None else str(value).strip()


def norm_name(value):
    name = re.sub(r"[^a-z0-9]+", "_", norm_text(value).lower())
    return re.sub(r"_+", "_", name).strip("_")


def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_sheet_name(name, used_names):
    name = re.sub(r"[\[\]:*?/\\]", "_", norm_text(name))
    name = re.sub(r"\s+", "_", name).strip("_") or "Sheet"
    name = name[:31]

    base = name
    suffix_number = 2
    while name in used_names:
        suffix = f"_{suffix_number}"
        name = f"{base[:31 - len(suffix)]}{suffix}"
        suffix_number += 1

    used_names.add(name)
    return name


def autosize_ws(worksheet, min_width=10, max_width=40):
    for column_cells in worksheet.columns:
        if not column_cells:
            continue
        column_index = column_cells[0].column
        max_length = max(
            len("" if cell.value is None else str(cell.value)) for cell in column_cells
        )
        worksheet.column_dimensions[get_column_letter(column_index)].width = max(
            min_width,
            min(max_width, max_length + 2),
        )


def style_header(worksheet, row_number=1, fill_color="1F4E78"):
    fill = PatternFill("solid", fgColor=fill_color)
    font = Font(bold=True, color="FFFFFF")
    for cell in worksheet[row_number]:
        cell.fill = fill
        cell.font = font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )


def read_csv_any(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        return pd.read_csv(path, encoding="latin1")


def read_excel_all_sheets(path):
    workbook = pd.ExcelFile(path)
    sheets = {}
    for sheet_name in workbook.sheet_names:
        try:
            sheets[sheet_name] = pd.read_excel(path, sheet_name=sheet_name)
        except Exception as exc:
            print(f"Skipping unreadable sheet {sheet_name} in {path.name}: {exc}")
    return sheets


def is_image(path):
    return path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}


def is_markdown(path):
    return path.suffix.lower() in {".md", ".txt"}


def is_tabular(path):
    return path.suffix.lower() in {".xlsx", ".xlsm", ".xls", ".csv"}


def choose_primary_workbook(candidates):
    if not candidates:
        return None

    def score(path):
        stem = norm_name(path.stem)
        value = 0
        for index, preferred_stem in enumerate(
            reversed(PREFERRED_MASTER_STEMS), start=1
        ):
            if preferred_stem in stem:
                value += 100 * index
        if path.suffix.lower() in {".xlsx", ".xlsm", ".xls"}:
            value += 50
        if "full" in stem:
            value += 20
        if "results" in stem:
            value += 10
        try:
            value += int(path.stat().st_mtime / 100000)
        except OSError:
            pass
        return value

    return max(candidates, key=score)


def canonical_table_key_from_file(path):
    return norm_name(path.stem)


def canonical_table_key_from_sheet(workbook_path, sheet_name):
    return f"{norm_name(workbook_path.stem)}__{norm_name(sheet_name)}"


def write_df_to_sheet(workbook, dataframe, sheet_name):
    worksheet = workbook.create_sheet(title=sheet_name)
    if dataframe is None or len(dataframe.columns) == 0:
        worksheet["A1"] = "Empty table"
        return worksheet

    dataframe_to_write = dataframe.copy().where(pd.notna(dataframe), "")
    worksheet.append(list(dataframe_to_write.columns))
    for row in dataframe_to_write.itertuples(index=False, name=None):
        worksheet.append(list(row))

    style_header(worksheet)
    worksheet.freeze_panes = "A2"
    autosize_ws(worksheet)
    return worksheet


def add_markdown_sheet(workbook, path, used_sheet_names):
    worksheet = workbook.create_sheet(
        title=safe_sheet_name("Summary_MD", used_sheet_names)
    )
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = path.read_text(encoding="utf-8-sig")
    except Exception:
        text = path.read_text(errors="replace")

    worksheet["A1"] = "Source file"
    worksheet["B1"] = str(path)
    worksheet["A2"] = "Content"
    style_header(worksheet)
    worksheet["A2"].font = Font(bold=True)

    for row_number, line in enumerate(text.splitlines(), start=3):
        worksheet.cell(row_number, 1).value = line

    worksheet.column_dimensions["A"].width = 140
    return worksheet


def add_figures_sheet(workbook, image_paths, used_sheet_names):
    if not image_paths:
        return None

    worksheet = workbook.create_sheet(
        title=safe_sheet_name("Figures", used_sheet_names)
    )
    worksheet["A1"] = "Embedded figures"
    style_header(worksheet)

    row_anchor = 3
    for image_path in sorted(image_paths):
        worksheet.cell(row_anchor, 1).value = image_path.name
        worksheet.cell(row_anchor, 1).font = Font(bold=True)
        worksheet.cell(row_anchor + 1, 1).value = str(image_path)
        worksheet.cell(row_anchor + 1, 1).alignment = Alignment(wrap_text=True)

        try:
            image = XLImage(str(image_path))
            max_width = 1100
            if image.width > max_width:
                scale = max_width / image.width
                image.width = int(image.width * scale)
                image.height = int(image.height * scale)
            worksheet.add_image(image, f"A{row_anchor + 3}")
            row_anchor += max(24, int(image.height / 20) + 6)
        except Exception as exc:
            worksheet.cell(row_anchor + 3, 1).value = f"[Could not embed image: {exc}]"
            row_anchor += 8

    worksheet.column_dimensions["A"].width = 140
    return worksheet


def add_index_sheet(workbook, inventory_df, included_df, used_sheet_names):
    worksheet = workbook.create_sheet(
        title=safe_sheet_name("Index", used_sheet_names),
        index=0,
    )
    worksheet["A1"] = "Conflict Analysis Outputs Consolidation Index"
    worksheet["A1"].font = Font(bold=True, size=14)
    worksheet["A3"] = "Root folder"
    worksheet["B3"] = str(OUTPUT_DIR)
    worksheet["A5"] = "Included tables"
    worksheet["A5"].font = Font(bold=True)

    included_columns = ["sheet_name", "source_type", "source_path", "n_rows", "n_cols"]
    worksheet.append(included_columns)
    for cell in worksheet[6]:
        cell.font = Font(bold=True)

    for row in included_df[included_columns].itertuples(index=False, name=None):
        worksheet.append(list(row))

    row_number = worksheet.max_row + 3
    worksheet.cell(row_number, 1).value = "Full file inventory"
    worksheet.cell(row_number, 1).font = Font(bold=True)
    row_number += 1

    inventory_columns = ["file_name", "suffix", "category", "size_bytes", "path"]
    for column_number, column_name in enumerate(inventory_columns, start=1):
        worksheet.cell(row_number, column_number).value = column_name
        worksheet.cell(row_number, column_number).font = Font(bold=True)
    row_number += 1

    for row in inventory_df[inventory_columns].itertuples(index=False, name=None):
        for column_number, value in enumerate(row, start=1):
            worksheet.cell(row_number, column_number).value = value
        row_number += 1

    autosize_ws(worksheet, min_width=12, max_width=60)
    return worksheet


if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Output folder not found: {OUTPUT_DIR}")

all_files = [path for path in OUTPUT_DIR.rglob("*") if path.is_file()]
source_files = [
    path
    for path in all_files
    if path.name.lower() not in IGNORE_FILE_NAMES
    and norm_name(path.stem) not in IGNORE_FILE_STEMS
]

inventory = []
for path in sorted(source_files):
    suffix = path.suffix.lower()
    if is_tabular(path):
        category = "tabular"
    elif is_image(path):
        category = "figure"
    elif is_markdown(path):
        category = "summary_text"
    elif suffix == ".parquet":
        category = "cache"
    elif suffix == ".zip":
        category = "archive"
    else:
        category = "other"

    inventory.append(
        {
            "file_name": path.name,
            "suffix": suffix,
            "category": category,
            "size_bytes": path.stat().st_size,
            "path": str(path),
        }
    )

inventory_df = pd.DataFrame(
    inventory,
    columns=["file_name", "suffix", "category", "size_bytes", "path"],
)
print("Found source files:", len(source_files))
print(inventory_df["category"].value_counts(dropna=False).to_dict())

xlsx_like = [
    path for path in source_files if path.suffix.lower() in {".xlsx", ".xlsm", ".xls"}
]
primary_workbook = choose_primary_workbook(xlsx_like)
print("Primary workbook:", primary_workbook if primary_workbook else "[None]")

master_path = OUTPUT_DIR / MASTER_WORKBOOK_NAME
backup_zip_path = OUTPUT_DIR / BACKUP_ZIP_NAME
workbook = Workbook()
workbook.remove(workbook.active)

used_sheet_names = set()
included_rows = []
existing_table_keys = set()

if primary_workbook is not None:
    try:
        primary_sheets = read_excel_all_sheets(primary_workbook)
        for original_sheet_name, dataframe in primary_sheets.items():
            sheet_name = safe_sheet_name(original_sheet_name, used_sheet_names)
            write_df_to_sheet(workbook, dataframe, sheet_name)
            existing_table_keys.update(
                {
                    canonical_table_key_from_sheet(
                        primary_workbook, original_sheet_name
                    ),
                    norm_name(original_sheet_name),
                    norm_name(f"{primary_workbook.stem}_{original_sheet_name}"),
                }
            )
            included_rows.append(
                {
                    "sheet_name": sheet_name,
                    "source_type": "primary_workbook_sheet",
                    "source_path": str(primary_workbook),
                    "n_rows": 0 if dataframe is None else len(dataframe),
                    "n_cols": 0 if dataframe is None else len(dataframe.columns),
                }
            )
    except Exception as exc:
        print(f"Could not import primary workbook {primary_workbook}: {exc}")

tabular_files = [
    path for path in source_files if is_tabular(path) and path != primary_workbook
]
grouped_files = {}
for path in tabular_files:
    grouped_files.setdefault(canonical_table_key_from_file(path), []).append(path)

selected_tabular = []
for files in grouped_files.values():
    selected_tabular.append(
        min(
            files,
            key=lambda path: (
                0 if path.suffix.lower() in {".xlsx", ".xlsm", ".xls"} else 1,
                path.name.lower(),
            ),
        )
    )

for path in sorted(selected_tabular, key=lambda item: item.name.lower()):
    if norm_name(path.stem) in existing_table_keys:
        continue

    try:
        if path.suffix.lower() == ".csv":
            dataframe = read_csv_any(path)
            sheet_name = safe_sheet_name(path.stem, used_sheet_names)
            write_df_to_sheet(workbook, dataframe, sheet_name)
            included_rows.append(
                {
                    "sheet_name": sheet_name,
                    "source_type": "csv",
                    "source_path": str(path),
                    "n_rows": len(dataframe),
                    "n_cols": len(dataframe.columns),
                }
            )

        elif path.suffix.lower() in {".xlsx", ".xlsm", ".xls"}:
            sheet_map = read_excel_all_sheets(path)
            if len(sheet_map) == 1:
                original_sheet_name, dataframe = next(iter(sheet_map.items()))
                if (
                    norm_name(path.stem) in existing_table_keys
                    or norm_name(original_sheet_name) in existing_table_keys
                ):
                    continue

                sheet_name = safe_sheet_name(path.stem, used_sheet_names)
                write_df_to_sheet(workbook, dataframe, sheet_name)
                included_rows.append(
                    {
                        "sheet_name": sheet_name,
                        "source_type": "single_sheet_workbook",
                        "source_path": str(path),
                        "n_rows": len(dataframe),
                        "n_cols": len(dataframe.columns),
                    }
                )
            else:
                for original_sheet_name, dataframe in sheet_map.items():
                    compound_key = canonical_table_key_from_sheet(
                        path, original_sheet_name
                    )
                    if (
                        compound_key in existing_table_keys
                        or norm_name(original_sheet_name) in existing_table_keys
                    ):
                        continue

                    sheet_name = safe_sheet_name(
                        f"{path.stem}_{original_sheet_name}",
                        used_sheet_names,
                    )
                    write_df_to_sheet(workbook, dataframe, sheet_name)
                    included_rows.append(
                        {
                            "sheet_name": sheet_name,
                            "source_type": "multi_sheet_workbook",
                            "source_path": f"{path} :: {original_sheet_name}",
                            "n_rows": len(dataframe),
                            "n_cols": len(dataframe.columns),
                        }
                    )
    except Exception as exc:
        print(f"Skipping tabular file {path.name}: {exc}")

included_df = pd.DataFrame(included_rows)
summary_candidates = [
    path
    for path in source_files
    if is_markdown(path)
    and (
        "summary" in path.name.lower()
        or "conflict_analysis_summary" in path.name.lower()
    )
]
if summary_candidates:
    add_markdown_sheet(workbook, sorted(summary_candidates)[0], used_sheet_names)

image_paths = [path for path in source_files if is_image(path)]
add_figures_sheet(workbook, image_paths, used_sheet_names)

cache_files = [path for path in source_files if path.suffix.lower() == ".parquet"]
other_files = [
    path
    for path in source_files
    if path.suffix.lower()
    not in {
        ".csv",
        ".xlsx",
        ".xlsm",
        ".xls",
        ".png",
        ".jpg",
        ".jpeg",
        ".webp",
        ".md",
        ".txt",
        ".parquet",
    }
]

if cache_files or other_files:
    notes_sheet = safe_sheet_name("Other_Files", used_sheet_names)
    notes_ws = workbook.create_sheet(title=notes_sheet)
    notes_ws["A1"] = "Files not merged as regular sheets"
    style_header(notes_ws)

    notes_ws.append([])
    notes_ws.append(["Type", "Path", "Reason"])
    for cell in notes_ws[3]:
        cell.font = Font(bold=True)

    for path in cache_files:
        notes_ws.append(
            [
                "cache",
                str(path),
                "Parquet cache; a workbook-friendly version may be included elsewhere.",
            ]
        )
    for path in other_files:
        notes_ws.append([path.suffix.lower(), str(path), "Not merged into workbook."])

    autosize_ws(notes_ws, min_width=12, max_width=90)

if included_df.empty:
    included_df = pd.DataFrame(
        columns=["sheet_name", "source_type", "source_path", "n_rows", "n_cols"]
    )
add_index_sheet(workbook, inventory_df, included_df, used_sheet_names)

workbook.save(master_path)
print("Saved consolidated workbook to:")
print(master_path)

with zipfile.ZipFile(backup_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in source_files:
        archive.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))

print("Saved backup zip to:")
print(backup_zip_path)

if MOVE_ORIGINALS_TO_ARCHIVE_SUBFOLDER:
    archive_dir = ensure_dir(OUTPUT_DIR / ARCHIVE_SUBFOLDER_NAME)
    for path in source_files:
        destination = archive_dir / path.name
        if destination.exists():
            suffix_number = 2
            while (archive_dir / f"{path.stem}_{suffix_number}{path.suffix}").exists():
                suffix_number += 1
            destination = archive_dir / f"{path.stem}_{suffix_number}{path.suffix}"
        shutil.move(str(path), str(destination))
    print("Moved original source files into archive folder:")
    print(archive_dir)

print("\nSummary:")
print("Output folder:", OUTPUT_DIR)
print("Consolidated workbook:", master_path.name)
print("Backup zip:", backup_zip_path.name)
print("Included sheets:", len(included_df))
print("Figures embedded:", len(image_paths))
print("Source files inventoried:", len(inventory_df))